In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

from datetime import datetime
from typing import Any, Dict, List

from dotenv import load_dotenv

# LangChain 관련 임포트
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LangSmith 임포트
from langsmith import Client
from langsmith.run_helpers import traceable

# 환경설정
load_dotenv()

# langSmith 추적 활성화
os.environ['LANGCHAIN_TRACING_V2']='true'
os.environ['LANGCHAIN_PROJECT']='langsmith_prj_001'
print(f"프로젝트 : {os.getenv('LANCHAIN_PROJECT')}")

llm = ChatOpenAI(model='gpt-4o-mini')
prompt = ChatPromptTemplate.from_template('{question}에 대해서 간단하게 설명해주세요')
chain = prompt | llm | StrOutputParser()

test_qeustion = [
    'AI', 'LangSmith', '대한민국'
]
for q in test_qeustion:
    response = chain.invoke({'question':q})
    print(f"질문 : {q}\n답변 : {response}")


# @traceable로 설정된 함수는 자동으로 추적
@traceable(name = f"custom_qa_function_{os.getenv('LANGCHAIN_PROJECT')}")
def answer_question(question:str)->str:
    prompt = f'다음 질문에 대해서 100자 이내로 요약해서 답변해주세요 : {question}'
    chain = llm | StrOutputParser()
    return chain.invoke(prompt)

result = answer_question('프로그래밍 전문가가 되는 방법 및 가이드')
print(f'answer_question : {result}')

# client 직접 사용
client = Client()

# 프로젝트 목록 조회
project_lists = list(client.list_projects())
for project in project_lists:
    print(project.name)


# 데이터 셋 생성
client.create_dataset(
    dataset_name=os.getenv('LANGCHAIN_PROJECT') + '_001',
    description=os.getenv('LANGCHAIN_PROJECT')+'_QA 평가용 데이터셋'
)


    # 평가용 예제
    examples = [
        {
            "inputs": {"question": "Python이란 무엇인가요?"},
            "outputs": {"answer": "Python은 프로그래밍 언어입니다."}
        },
        {
            "inputs": {"question": "1+1은?"},
            "outputs": {"answer": "2입니다."}
        },
        {
            "inputs": {"question": "AI란?"},
            "outputs": {"answer": "인공지능입니다."}
        }
    ]




In [1]:
# LangSmith API를 이용한 LLM 모니터링
import os
import warnings
warnings.filterwarnings("ignore")

from datetime import datetime
from typing import Any, Dict, List

from dotenv import load_dotenv

# LangChain 관련 임포트
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LangSmith 임포트
from langsmith import Client
from langsmith.run_helpers import traceable

# 환경설정
load_dotenv()

# langsmith 추적 활성화
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = 'langsmith_prj_001'
print(f"프로젝트 : {os.getenv('LANGCHAIN_PROJECT')}")

llm = ChatOpenAI(model = 'gpt-4o-mini')
prompt = ChatPromptTemplate.from_template('{question}에대해서 간단하게 설명해주세요')
chain = prompt | llm | StrOutputParser()

test_questions = [
    'AI','LangSmith','대한민국'
]
for q in test_questions:
    response = chain.invoke({'question':q})
    print(f'질문 : {q}  답변 : {response}')


# @traceable 로 설정된 함수는 자동으로 추적
@traceable(name=f'custom_qa_function_{os.getenv('LANGCHAIN_PROJECT')}')
def answer_question(question:str)->str:
    prompt = f'다음 질문에 대해서 100자 이내로 요약해서 답변해주세요 : {question}'
    chain = llm | StrOutputParser()
    return chain.invoke(prompt)

result = answer_question('프로그래밍 전문가가 되는 방법 및 가이드')
print(f'answer_question : {result}')

# client 직접사용
client = Client()
# 프로젝트 목록조회
print('프로젝트 목록조회')
project_lists = list(client.list_projects())
for project in project_lists:
    print(project.name)


# 데이터 셋 생성
dataset_name=f"{os.getenv('LANGCHAIN_PROJECT')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description=os.getenv('LANGCHAIN_PROJECT') + '_QA 평가용 데이터셋'
)
# 평가용 예제
examples = [
    {
        "inputs": {"question": "Python이란 무엇인가요?"},
        "outputs": {"answer": "Python은 프로그래밍 언어입니다."}
    },
    {
        "inputs": {"question": "1+1은?"},
        "outputs": {"answer": "2입니다."}
    },
    {
        "inputs": {"question": "AI란?"},
        "outputs": {"answer": "인공지능입니다."}
    }
]
for ex in examples:
    client.create_example(
        inputs=ex['inputs'],
        outputs=ex['outputs'],
        dataset_id=dataset.id
    )
print(f'    {len(examples)}개 예제 추가 완료')

from langsmith.evaluation import evaluate
client = Client()
# 평가모델 정의
llm = ChatOpenAI(model='gpt-4o-mini',temperature=0)
#평가 함수 실행
def predict(inputs:str)->Dict[str,str]:
    q = inputs['question']
    result = llm.invoke(f'{q} 간단히 답해줘')
    return {'answer':result.content}
def simple_correctness(run, example):
    """run.outputs 로 모델 답변을 가져오는 방식"""

    gold = example.outputs["answer"]
    pred = run.outputs["answer"]

    score = 1.0 if gold in pred else 0.0

    return {
        "key": "correctness",
        "score": score,
        "comment": f"gold={gold} | pred={pred}"
    }
# 평가실행
results = evaluate(
    predict,
    data=dataset_name,            
    evaluators=[simple_correctness]
)

print('\n평가 결과 요약')
print(results)



# 정리 (테스트 후 삭제)
# client.delete_dataset(dataset_id=dataset.id)
# print(' 데이터셋 삭제완료')

프로젝트 : langsmith_prj_001
질문 : AI  답변 : AI(인공지능)는 인간의 지능을 모방하거나 그에 준하는 기능을 수행할 수 있는 컴퓨터 시스템이나 프로그램을 의미합니다. AI는 다양한 기술과 알고리즘을 활용하여 데이터를 분석하고, 학습하며, 결정을 내리는 능력을 갖추고 있습니다.

AI는 크게 두 가지로 나눌 수 있습니다:

1. **약한 AI(Weak AI)**: 특정 작업을 수행하도록 설계된 시스템으로, 예를 들어, 음성 인식, 이미지 인식, 추천 시스템 등이 이에 해당합니다. 이러한 시스템은 인간의 사고 과정을 완전히 이해하지 못하지만, 주어진 특정 문제를 해결하는 데 뛰어난 성능을 보입니다.

2. **강한 AI(Strong AI)**: 인간과 유사한 수준의 지능을 갖춘 시스템을 의미합니다. 강한 AI는 스스로 사고하고, 이해하며, 새로운 문제를 해결할 수 있는 능력을 갖출 것으로 기대되지만, 현재까지는 이론적인 개념에 가까운 상태입니다.

AI의 응용 분야는 매우 다양하며, 의료, 금융, 제조업, 자율주행차, 고객 서비스 등에서 활용되고 있습니다. AI 기술이 발전함에 따라 우리의 생활은 점점 더 많은 변화와 혜택을 누리게 될 것입니다.
질문 : LangSmith  답변 : LangSmith는 자연어 처리(NLP) 및 인공지능(AI) 기술을 활용하여 언어 관련 문제를 해결하는 플랫폼이나 도구를 지칭할 수 있습니다. 일반적으로 이러한 플랫폼은 텍스트 분석, 언어 번역, 대화형 AI 시스템 등을 지원하며, 개발자와 기업들이 AI 기반의 언어 서비스를 쉽게 구현하도록 돕습니다.

LangSmith의 구체적인 기능이나 서비스는 해당 플랫폼의 공식 웹사이트나 문서에서 더 자세히 확인할 수 있습니다.
질문 : 대한민국  답변 : 대한민국, 공식적으로는 대한민국(大韓民國, Republic of Korea)은 동아시아 한반도의 남부에 위치한 국가입니다. 북쪽으로는 조선민주주의인민공화국(북한)과 국경을 접하고 있으며, 서쪽으로는 황해, 동쪽으로는 

3it [00:05,  1.88s/it]


평가 결과 요약
<ExperimentResults crazy-exchange-2>
